# Home Credit Default Risk Prediction

**Goal:** predict the probability that a loan applicant will have difficulty repaying a loan (`TARGET = 1`).

This notebook provides a reproducible baseline workflow:
1. Load `application_train.csv` and `application_test.csv` (or files named `train.csv` and `test.csv`).
2. Inspect the data and target balance.
3. Split the training data into train/validation sets.
4. Preprocess numeric and categorical columns without leaking validation data.
5. Train a Logistic Regression baseline and evaluate ROC-AUC.
6. Fit on all training rows and create `submission.csv`.

> **Important:** This notebook has not been run against your competition dataset in this environment. Run all cells in Colab to generate fresh validation metrics and a submission. Do not claim a score unless it appears in your run output.

## Dataset
Use the competition-provided data and follow the competition/course rules. Do not commit private or restricted dataset files to a public repository.

## 1. Imports and configuration

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TARGET = "TARGET"
ID_COL = "SK_ID_CURR"

warnings.filterwarnings("ignore", category=FutureWarning)
print("Imports successful.")

## 2. Load the data

In Google Colab, this cell asks you to upload the train and test CSV files. The filenames should start with `train` and `test` (for example, `train.csv` and `test.csv`). If you are running locally, place both CSVs in the notebook's current folder and set `USE_COLAB_UPLOAD = False`.

In [ ]:
USE_COLAB_UPLOAD = True  # Change to False if the CSV files are already in the current folder.

if USE_COLAB_UPLOAD:
    try:
        from google.colab import files
        uploaded = files.upload()
        available_files = list(uploaded.keys())
    except ImportError:
        raise RuntimeError("Colab upload is unavailable. Set USE_COLAB_UPLOAD = False and place CSVs beside this notebook.")
else:
    available_files = [p.name for p in Path(".").glob("*.csv")]

train_matches = [f for f in available_files if Path(f).name.lower().startswith("train")]
test_matches = [f for f in available_files if Path(f).name.lower().startswith("test")]

if not train_matches or not test_matches:
    raise FileNotFoundError(
        "Could not identify both CSV files. Upload files named like train.csv and test.csv."
    )

train_path, test_path = train_matches[0], test_matches[0]
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

if TARGET not in train.columns:
    raise ValueError(f"Training data must contain the target column {TARGET!r}.")
if ID_COL not in train.columns or ID_COL not in test.columns:
    raise ValueError(f"Both files must contain the ID column {ID_COL!r}.")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
display(train.head())

## 3. Basic data checks

In [ ]:
print("Target distribution:")
display(train[TARGET].value_counts(dropna=False).rename("count").to_frame())
print("\nMissing values: top 15 training columns")
display(train.isna().mean().sort_values(ascending=False).head(15).rename("missing_fraction").to_frame())

if train[TARGET].isna().any():
    raise ValueError("TARGET contains missing values; inspect the data before training.")
if not set(train[TARGET].unique()).issubset({0, 1}):
    raise ValueError("This notebook expects a binary TARGET containing only 0 and 1.")

## 4. Prepare features and a validation split

The ID column is excluded from model features. The preprocessing pipeline is fitted only on the training split, which helps avoid validation leakage.

In [ ]:
y = train[TARGET].astype(int)
X = train.drop(columns=[TARGET, ID_COL], errors="ignore")
X_test = test.drop(columns=[ID_COL], errors="ignore")

# Align columns in case train and test columns differ.
X_test = X_test.reindex(columns=X.columns)

categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_cols = [c for c in X.columns if c not in categorical_cols]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print(f"Numeric columns: {len(numeric_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")
print(f"Train split: {X_train.shape}; validation split: {X_valid.shape}")

## 5. Train the baseline and evaluate ROC-AUC

In [ ]:
baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

baseline_model.fit(X_train, y_train)
valid_prob = baseline_model.predict_proba(X_valid)[:, 1]
validation_auc = roc_auc_score(y_valid, valid_prob)

print(f"Validation ROC-AUC: {validation_auc:.5f}")

## 6. Optional: try LightGBM

Gradient-boosting models often work well on tabular credit-risk data. This optional cell tries LightGBM if it is installed; it does not silently replace the baseline. In Colab, you may install it with `!pip install lightgbm` in a separate cell, then rerun this cell. Compare validation ROC-AUC values using the same split before choosing a model.

In [ ]:
try:
    from lightgbm import LGBMClassifier

    lgbm_preprocessor = ColumnTransformer(transformers=[
        ("numeric", SimpleImputer(strategy="median"), numeric_cols),
        ("categorical", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_cols)
    ])

    lgbm_model = Pipeline(steps=[
        ("preprocessor", lgbm_preprocessor),
        ("classifier", LGBMClassifier(
            n_estimators=500,
            learning_rate=0.03,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            verbosity=-1
        ))
    ])
    lgbm_model.fit(X_train, y_train)
    lgbm_valid_prob = lgbm_model.predict_proba(X_valid)[:, 1]
    lgbm_auc = roc_auc_score(y_valid, lgbm_valid_prob)
    print(f"LightGBM validation ROC-AUC: {lgbm_auc:.5f}")
    if lgbm_auc > validation_auc:
        print("LightGBM performed better on this validation split.")
    else:
        print("The baseline matched or outperformed LightGBM on this split.")
except ImportError:
    print("LightGBM is not installed. Baseline results are still available.")
    print("To try it in Colab, run: !pip install lightgbm")
    lgbm_model = None

## 7. Train the selected model on all training rows and create a submission

By default, the baseline is used so this notebook runs without an optional LightGBM dependency. If LightGBM is installed and performed better on validation, set `USE_LIGHTGBM = True` below. Refit the chosen pipeline on all training rows before predicting the test set.

In [ ]:
USE_LIGHTGBM = False  # Set True only if LightGBM was available and had a better validation ROC-AUC.

if USE_LIGHTGBM:
    if lgbm_model is None:
        raise RuntimeError("LightGBM is unavailable. Install it and rerun the optional LightGBM cell first.")
    final_model = lgbm_model
else:
    final_model = baseline_model

final_model.fit(X, y)
test_prob = final_model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET: test_prob
})
output_path = Path("submission.csv")
submission.to_csv(output_path, index=False)

print("Saved:", output_path.resolve())
print("Submission shape:", submission.shape)
print("Prediction range:", float(submission[TARGET].min()), "to", float(submission[TARGET].max()))
display(submission.head())

## 8. Download the submission (Google Colab only)

In [ ]:
try:
    from google.colab import files
    files.download("submission.csv")
except ImportError:
    print("submission.csv is saved in the current working directory.")

## 9. Notes and next improvements

- Record the validation ROC-AUC produced by your own run; it can vary with data and environment.
- Compare models on the same validation split and avoid selecting a model based on the public leaderboard alone.
- Potential next steps: cross-validation, careful feature engineering, hyperparameter tuning, and calibration checks.
- Respect the competition's data-use and submission rules.
- This is an educational baseline, not a production credit-decision system.